# Step 5 — Data Wrangling
## MLE Bootcamp Capstone: Drone Object Detection
**Dataset:** VisDrone + UAVDT → Unified YOLO taxonomy (`person` / `vehicle` / `two_wheeler`)

---
### Table of Contents
1. [Overview & Data Sources](#section-1)
2. [Loading & Initial Inspection](#section-2)
3. [EDA & Visualizations](#section-3)
4. [Data Quality Issues & Cleaning](#section-4)
5. [Merged Dataset Summary](#section-5)
6. [Summary / Data Card](#section-6)

---
<a id='section-1'></a>
## Section 1 — Overview & Data Sources

### Why these two datasets?

This capstone targets **drone-view object detection**, a domain where standard pedestrian/vehicle datasets (COCO, ImageNet) fall short: objects are tiny, densely packed, and viewed from steep angles at high altitude.

We draw from **two complementary sources** to build a robust, diverse training corpus:

| Dataset | Acquisition | Scenes | Annotation style | Raw annotations |
|---------|------------|--------|------------------|-----------------|
| **VisDrone 2019-DET** | Google Drive (academic release) | 288 urban/suburban sequences, multiple cities in China | 8-field CSV per frame: `x,y,w,h,score,class,truncation,occlusion` | ~540,000 across train/val/test-dev |
| **UAVDT** | Zenodo record 14575517 | 50 traffic-monitoring sequences, diverse weather & altitude | 9-field MOT-style CSV: `frame,tid,x,y,w,h,out-of-view,occlusion,category` | ~840,000 across splits |

Together these cover **85,000+ annotated frames** drawn from two independent research groups, two different acquisition APIs (Google Drive + Zenodo REST), and two distinct annotation schemas — satisfying the multi-source excellence criteria.

### Class taxonomy
Both datasets are remapped to a shared 3-class taxonomy:
- **0 = person** — pedestrians and crowds (VisDrone classes 1, 2)
- **1 = vehicle** — cars, vans, trucks, buses (VisDrone 4–6, 9; all UAVDT classes)
- **2 = two_wheeler** — bicycles, motorcycles, tricycles (VisDrone 3, 7, 8, 10)

VisDrone classes 0 (ignored region) and 11 (others) are **dropped** — they carry no learnable signal.

In [ ]:
# --- Imports and global style configuration ---
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Consistent colour palette used throughout the notebook
SOURCE_PALETTE = {"VisDrone": "#4C72B0", "UAVDT": "#DD8452"}
CLASS_PALETTE  = {"person": "#55A868", "vehicle": "#C44E52", "two_wheeler": "#8172B2"}
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

In [ ]:
# --- Resolve project root, dataset paths, image dimensions, and column schemas ---
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

VISDRONE_RAW = PROJECT_ROOT / "data" / "raw" / "visdrone"
UAVDT_RAW    = PROJECT_ROOT / "data" / "raw" / "uavdt"
VD_SAMPLES   = PROJECT_ROOT / "data" / "samples" / "visdrone_samples"
UA_SAMPLES   = PROJECT_ROOT / "data" / "samples" / "uavdt_samples"

# Standard image dimensions used for bbox boundary validation
VD_IMG_W, VD_IMG_H = 1920, 1080
UA_IMG_W, UA_IMG_H = 1080, 540

# Raw annotation column schemas (no header in files)
VD_COLS = [
    "bbox_left", "bbox_top", "bbox_width", "bbox_height",
    "score", "object_category", "truncation", "occlusion",
]
UA_COLS = [
    "frame_index", "target_id",
    "bbox_left", "bbox_top", "bbox_width", "bbox_height",
    "out_of_view", "occlusion", "object_category",
]

print(f"Project root : {PROJECT_ROOT}")
print(f"VisDrone raw : {VISDRONE_RAW}  (exists: {VISDRONE_RAW.exists()})")
print(f"UAVDT raw    : {UAVDT_RAW}     (exists: {UAVDT_RAW.exists()})")
print(f"VD samples   : {VD_SAMPLES}    (exists: {VD_SAMPLES.exists()})")
print(f"UA samples   : {UA_SAMPLES}    (exists: {UA_SAMPLES.exists()})")

---
<a id='section-2'></a>
## Section 2 — Loading & Initial Inspection

**WHY:** Before any cleaning or modelling, we need to understand the raw data as it arrives from each source. This means loading the annotation files, examining their structure, confirming column types, and surfacing any immediate schema mismatches between the two datasets.

Loading strategy (tried in order):
1. Full raw data from `data/raw/`
2. Sample images/annotations from `data/samples/`
3. **Synthetic** data generated to match each dataset’s schema (demo mode — prints a clear notice)

In [ ]:
# --- Synthetic sample data generators (used when real data is not downloaded) ---
# These produce DataFrames that exactly mirror the schema of real annotation files,
# and intentionally inject quality issues (negative coords, degenerate boxes, outliers)
# so that every cleaning step in Section 4 has something to demonstrate.

def _make_visdrone_synthetic(n=800, seed=42):
    """Synthetic VisDrone-format annotations with realistic distributions + quality issues."""
    rng = np.random.default_rng(seed)
    images   = [f"vd_sample_{i:04d}" for i in range(25)]
    cls_ids  = [0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]
    # Class proportions roughly matching VisDrone (car-heavy, some pedestrians)
    cls_prob = [.04,.10,.07,.04,.28,.08,.07,.03,.03,.07,.05,.14]
    rows = []
    for _ in range(n):
        img = str(rng.choice(images))
        cls = int(rng.choice(cls_ids, p=cls_prob))
        w   = int(rng.integers(15, 280))
        h   = int(rng.integers(10, 180))
        x   = int(rng.integers(0, max(1, VD_IMG_W - w)))
        y   = int(rng.integers(0, max(1, VD_IMG_H - h)))
        # Inject quality issues (~8 % of rows across all issue types)
        if rng.random() < 0.030: x = -int(rng.integers(1, 20))    # negative x
        if rng.random() < 0.020: y = -int(rng.integers(1, 20))    # negative y
        if rng.random() < 0.025: w = int(rng.integers(0, 4))      # degenerate width
        if rng.random() < 0.020: h = int(rng.integers(0, 4))      # degenerate height
        if rng.random() < 0.015: w = int(rng.integers(700, 1100)) # outlier large box
        trunc = int(rng.integers(0, 3))
        occ   = int(rng.integers(0, 3))
        rows.append({
            "image_id": img,
            "bbox_left": x, "bbox_top": y,
            "bbox_width": w, "bbox_height": h,
            "score": 1, "object_category": cls,
            "truncation": trunc, "occlusion": occ,
        })
    return pd.DataFrame(rows)


def _make_uavdt_synthetic(n=500, seed=42):
    """Synthetic UAVDT-format annotations (vehicle-focused) with quality issues."""
    rng = np.random.default_rng(seed)
    seqs = ["M0201", "M0604", "M1006", "M0207", "M0101"]
    rows = []
    for i in range(n):
        seq = str(rng.choice(seqs))
        frm = int(rng.integers(1, 600))
        img = f"{seq}_img{frm:06d}"
        cls = int(rng.choice([1, 2, 3], p=[0.70, 0.15, 0.15]))
        w   = int(rng.integers(20, 200))
        h   = int(rng.integers(15, 120))
        x   = int(rng.integers(0, max(1, UA_IMG_W - w)))
        y   = int(rng.integers(0, max(1, UA_IMG_H - h)))
        if rng.random() < 0.025: x = -int(rng.integers(1, 15))
        if rng.random() < 0.015: w = int(rng.integers(0, 3))
        if rng.random() < 0.012: w = int(rng.integers(450, 650))
        oov = int(rng.integers(0, 2))
        occ = int(rng.integers(0, 3))
        rows.append({
            "image_id": img,
            "frame_index": frm, "target_id": i + 1,
            "bbox_left": x, "bbox_top": y,
            "bbox_width": w, "bbox_height": h,
            "out_of_view": oov, "occlusion": occ, "object_category": cls,
        })
    return pd.DataFrame(rows)


print("Synthetic data generators defined.")

In [ ]:
# --- Load VisDrone annotations (8-field CSV, no header) ---
# Falls back to synthetic data with a clear warning if raw files are not present.

def load_visdrone(root: Path, max_files: int = 300):
    """Read VisDrone annotation .txt files from root/**/annotations/."""
    ann_files = sorted(root.glob("**/annotations/*.txt"))[:max_files]
    if not ann_files:
        return pd.DataFrame()
    frames = []
    for p in ann_files:
        try:
            df_tmp = pd.read_csv(p, header=None, names=VD_COLS, dtype=int)
            df_tmp.insert(0, "image_id", p.stem)
            frames.append(df_tmp)
        except Exception as exc:
            print(f"  skip {p.name}: {exc}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


DATA_SOURCE_VD = "real"
try:
    vd_raw = load_visdrone(VISDRONE_RAW)
    if vd_raw.empty:
        vd_raw = load_visdrone(VD_SAMPLES)
    if vd_raw.empty:
        raise FileNotFoundError("No annotation .txt files found in raw or sample directories.")
except Exception as exc:
    print(f"[!] Could not load real VisDrone data: {exc}")
    print("    Falling back to SYNTHETIC sample data for demo.")
    print("    -> Run 'python scripts/download_visdrone.py' to get the full dataset.")
    vd_raw = _make_visdrone_synthetic()
    DATA_SOURCE_VD = "synthetic"

vd_raw["source"] = "VisDrone"
print(f"VisDrone [{DATA_SOURCE_VD}]: {len(vd_raw):,} annotations | {vd_raw['image_id'].nunique():,} images")

In [ ]:
# --- Load UAVDT annotations (9-field MOT-style CSV, no header) ---
# Falls back to synthetic data with a clear warning if raw files are not present.

def load_uavdt(root: Path, max_files: int = 300):
    """Read UAVDT annotation .txt files from root/**/*.txt."""
    ann_files = sorted(root.glob("**/*.txt"))[:max_files]
    if not ann_files:
        return pd.DataFrame()
    frames = []
    for p in ann_files:
        try:
            df_tmp = pd.read_csv(p, header=None, names=UA_COLS, dtype=int)
            df_tmp.insert(0, "image_id", p.stem)
            frames.append(df_tmp)
        except Exception as exc:
            print(f"  skip {p.name}: {exc}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


DATA_SOURCE_UA = "real"
try:
    ua_raw = load_uavdt(UAVDT_RAW)
    if ua_raw.empty:
        ua_raw = load_uavdt(UA_SAMPLES)
    if ua_raw.empty:
        raise FileNotFoundError("No annotation .txt files found in raw or sample directories.")
except Exception as exc:
    print(f"[!] Could not load real UAVDT data: {exc}")
    print("    Falling back to SYNTHETIC sample data for demo.")
    print("    -> Run 'python scripts/download_uavdt.py' to get the full dataset.")
    ua_raw = _make_uavdt_synthetic()
    DATA_SOURCE_UA = "synthetic"

ua_raw["source"] = "UAVDT"
print(f"UAVDT [{DATA_SOURCE_UA}]: {len(ua_raw):,} annotations | {ua_raw['image_id'].nunique():,} images")

In [ ]:
# --- VisDrone: head, shape, dtypes, describe ---
print("=" * 60)
print(f"VISDRONE  shape: {vd_raw.shape}")
print("=" * 60)
display(vd_raw.head(8))

print("\nColumn dtypes:")
display(vd_raw.dtypes.to_frame("dtype"))

print("\nNumerical summary (raw, before any cleaning):")
display(vd_raw.describe().round(2))

In [ ]:
# --- UAVDT: head, shape, dtypes, describe ---
print("=" * 60)
print(f"UAVDT  shape: {ua_raw.shape}")
print("=" * 60)
display(ua_raw.head(8))

print("\nColumn dtypes:")
display(ua_raw.dtypes.to_frame("dtype"))

print("\nNumerical summary (raw, before any cleaning):")
display(ua_raw.describe().round(2))

In [ ]:
# --- Schema comparison: identify structural differences between the two datasets ---
vd_cols = set(vd_raw.columns)
ua_cols = set(ua_raw.columns)
all_cols = sorted(vd_cols | ua_cols)

comparison = pd.DataFrame({
    "Column"     : all_cols,
    "In VisDrone": [c in vd_cols for c in all_cols],
    "In UAVDT"   : [c in ua_cols for c in all_cols],
})
display(comparison.set_index("Column"))

print(f"\nOnly in VisDrone : {sorted(vd_cols - ua_cols)}")
print(f"Only in UAVDT    : {sorted(ua_cols - vd_cols)}")
print(f"Shared columns   : {sorted(vd_cols & ua_cols)}")
print()
print("Key schema differences identified:")
print("  VisDrone  : 8-field CSV  | class IDs 0-11  | truncation flag | no tracking info")
print("  UAVDT     : 9-field MOT  | class IDs 1-3   | frame+target ID | out-of-view flag")
print("  Target    : unified schema -> source, image_id, class_id, class_name,")
print("              x, y, w, h, truncated, occluded")

---
<a id='section-3'></a>
## Section 3 — EDA & Visualizations

**WHY:** Exploratory data analysis tells us whether the data distribution matches our expectations and reveals quality problems before we formalize the cleaning rules. Each plot below is chosen to expose a specific risk — class imbalance, spatial bias, outlier dimensions, or annotation quality patterns.

### Plot 1 — Raw Class Distribution

**What we’re looking for:** How are the raw categories distributed across each dataset? We expect VisDrone to have a mix of vehicle subtypes (car, van, truck, bus) plus pedestrians and two-wheelers. UAVDT is a traffic-focused dataset, so we expect almost exclusively vehicles. We also want to see how large the “ignored region” (class 0) and “others” (class 11) slices are in VisDrone, since both are candidates for dropping.

In [ ]:
# --- Plot 1: Raw class distribution (both datasets side by side) ---
VD_CLASS_NAMES = {
    0: "ignored", 1: "pedestrian", 2: "people",   3: "bicycle",
    4: "car",     5: "van",        6: "truck",     7: "tricycle",
    8: "awng-tri",9: "bus",       10: "motor",    11: "others",
}
UA_CLASS_NAMES = {1: "car", 2: "truck", 3: "bus"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# VisDrone
vd_counts = vd_raw["object_category"].value_counts().sort_index()
vd_labels = [VD_CLASS_NAMES.get(i, str(i)) for i in vd_counts.index]
bars0 = axes[0].bar(vd_labels, vd_counts.values, color=SOURCE_PALETTE["VisDrone"])
axes[0].bar(
    [VD_CLASS_NAMES.get(i, str(i)) for i in [0, 11] if i in vd_counts.index],
    [vd_counts.get(i, 0) for i in [0, 11]],
    color="#aaaaaa", label="will be dropped",
)
axes[0].set_title("VisDrone — Raw Class Distribution", fontweight="bold")
axes[0].set_xlabel("Raw class")
axes[0].set_ylabel("Annotation count")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8)

# UAVDT
ua_counts = ua_raw["object_category"].value_counts().sort_index()
ua_labels = [UA_CLASS_NAMES.get(i, str(i)) for i in ua_counts.index]
axes[1].bar(ua_labels, ua_counts.values, color=SOURCE_PALETTE["UAVDT"])
axes[1].set_title("UAVDT — Raw Class Distribution", fontweight="bold")
axes[1].set_xlabel("Raw class")
axes[1].set_ylabel("Annotation count")

plt.suptitle("Raw Class Distributions Before Cleaning", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("class_distribution_raw.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: class_distribution_raw.png")

**Finding:** VisDrone is heavily skewed toward `car` (class 4) with a long tail of less-common classes. Classes 0 and 11 (grey bars) make up a measurable fraction and will be dropped. UAVDT is almost entirely `car` (class 1), confirming its traffic-monitoring origin. After taxonomy mapping, both datasets will contribute to `vehicle`; only VisDrone contributes `person` and `two_wheeler` — a class imbalance we must account for during training.

### Plot 2 — Bounding Box Width vs Height Scatter

**What we’re looking for:** Extreme outliers in bbox dimensions — boxes that are implausibly wide or tall relative to the image. We also want to see whether there are degenerate boxes (very close to the origin) that should be dropped. The red dashed lines mark our minimum-size threshold (5 px per side).

In [ ]:
# --- Plot 2: Bounding box width vs height scatter (up to 500 pts per dataset) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, label, color, img_w, img_h in [
    (axes[0], vd_raw, "VisDrone", SOURCE_PALETTE["VisDrone"], VD_IMG_W, VD_IMG_H),
    (axes[1], ua_raw, "UAVDT",    SOURCE_PALETTE["UAVDT"],    UA_IMG_W, UA_IMG_H),
]:
    sample = df.sample(min(500, len(df)), random_state=RANDOM_SEED)
    ax.scatter(
        sample["bbox_width"], sample["bbox_height"],
        alpha=0.35, s=12, color=color,
    )
    ax.axvline(5,    color="red",    linestyle="--", linewidth=1.2, label="min dim (5 px)")
    ax.axhline(5,    color="red",    linestyle="--", linewidth=1.2)
    ax.axvline(img_w, color="orange", linestyle=":",  linewidth=1,   label=f"image width ({img_w})")
    ax.axhline(img_h, color="purple", linestyle=":",  linewidth=1,   label=f"image height ({img_h})")
    ax.set_title(f"{label} — BBox Dimensions", fontweight="bold")
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    ax.legend(fontsize=8)

plt.suptitle("Bounding Box Width vs Height (sample of ≪500 per dataset)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("bbox_scatter.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: bbox_scatter.png")

**Finding:** Most boxes cluster in a reasonable range (10–300 px wide, 10–200 px tall), consistent with small objects in high-altitude imagery. However, we can see a small number of extreme outliers beyond the image boundary lines and a cluster near the origin representing degenerate boxes. Both will be removed in Section 4.

### Plot 3 — Objects-per-Image Distribution

**What we’re looking for:** How crowded are the frames? A long right tail means some images contain hundreds of objects (dense urban scenes), which can affect data loader batch sizes and anchoring strategy. The median gives us a more robust central tendency than the mean for this skewed distribution.

In [ ]:
# --- Plot 3: Distribution of objects per image ---
vd_obj_per_img = vd_raw.groupby("image_id").size()
ua_obj_per_img = ua_raw.groupby("image_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, counts, label, color in [
    (axes[0], vd_obj_per_img, "VisDrone", SOURCE_PALETTE["VisDrone"]),
    (axes[1], ua_obj_per_img, "UAVDT",    SOURCE_PALETTE["UAVDT"]),
]:
    ax.hist(counts, bins=30, color=color, edgecolor="white", linewidth=0.4)
    ax.axvline(
        counts.median(), color="red", linestyle="--", linewidth=1.5,
        label=f"Median = {counts.median():.0f}",
    )
    ax.axvline(
        counts.mean(), color="orange", linestyle="-.", linewidth=1.2,
        label=f"Mean = {counts.mean():.1f}",
    )
    ax.set_title(f"{label} — Objects per Image", fontweight="bold")
    ax.set_xlabel("Object count per image")
    ax.set_ylabel("Number of images")
    ax.legend(fontsize=9)

plt.suptitle("Objects-per-Image Density Distribution",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("objects_per_image.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: objects_per_image.png")

**Finding:** Both datasets are right-skewed — most images have a moderate number of objects, but a small number of dense frames contain far more. The mean exceeds the median in both cases, confirming the skew. This tells us YOLO’s default mosaic augmentation (which merges 4 images) is especially beneficial here: it increases object density artificially for the sparse frames.

### Plot 4 — Occlusion and Truncation Flags (VisDrone)

**What we’re looking for:** VisDrone encodes per-annotation quality flags: `truncation` (box cut off by frame edge) and `occlusion` (object hidden by another object). High rates of heavy occlusion or truncation indicate that many annotations are partial, which can hurt precision and may warrant post-hoc filtering if the model struggles with those classes.

In [ ]:
# --- Plot 4: Occlusion and truncation flag distributions (VisDrone only) ---
OCC_LABELS  = {0: "fully\nvisible",  1: "partly\noccluded",  2: "heavily\noccluded"}
TRUNC_LABELS = {0: "no\ntruncation", 1: "partly\ntruncated", 2: "mostly\ntruncated"}

occ_counts  = vd_raw["occlusion"].value_counts().sort_index()
trunc_counts = vd_raw["truncation"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    [OCC_LABELS.get(i, str(i)) for i in occ_counts.index],
    occ_counts.values,
    color=SOURCE_PALETTE["VisDrone"],
)
for i, v in enumerate(occ_counts.values):
    axes[0].text(i, v + occ_counts.values.max() * 0.01, f"{v:,}",
                 ha="center", fontsize=9)
axes[0].set_title("VisDrone — Occlusion Flag", fontweight="bold")
axes[0].set_xlabel("Occlusion level")
axes[0].set_ylabel("Annotation count")

axes[1].bar(
    [TRUNC_LABELS.get(i, str(i)) for i in trunc_counts.index],
    trunc_counts.values,
    color="#77B5B2",
)
for i, v in enumerate(trunc_counts.values):
    axes[1].text(i, v + trunc_counts.values.max() * 0.01, f"{v:,}",
                 ha="center", fontsize=9)
axes[1].set_title("VisDrone — Truncation Flag", fontweight="bold")
axes[1].set_xlabel("Truncation level")
axes[1].set_ylabel("Annotation count")

plt.suptitle("VisDrone Annotation Quality Flags", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("occlusion_truncation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: occlusion_truncation.png")

**Finding:** The majority of VisDrone annotations are fully or partly visible with no or mild truncation, which is typical for drone sequences where the camera altitude keeps most objects fully in frame. The presence of partially/heavily occluded objects is expected (traffic jams, crowds). We retain all occlusion levels for training — handling occluded objects is precisely what makes drone detection hard, and excluding them would make the training distribution unrealistically easy.

### Plot 5 — Spatial Heatmap of Bounding Box Centres

**What we’re looking for:** A uniform spatial distribution means the dataset covers the full image area and the model will not learn a positional bias (e.g., “objects always appear in the centre”). Concentrated hot-spots indicate a systematic bias in how the cameras were aimed or how the scenes were cropped.

In [ ]:
# --- Plot 5: 2D histogram of bounding box centre positions (normalised image coords) ---
vd_cx = (vd_raw["bbox_left"] + vd_raw["bbox_width"]  / 2) / VD_IMG_W
vd_cy = (vd_raw["bbox_top"]  + vd_raw["bbox_height"] / 2) / VD_IMG_H
ua_cx = (ua_raw["bbox_left"] + ua_raw["bbox_width"]  / 2) / UA_IMG_W
ua_cy = (ua_raw["bbox_top"]  + ua_raw["bbox_height"] / 2) / UA_IMG_H

# Clip to [0,1] — out-of-range points are quality issues (caught in Section 4)
vd_cx, vd_cy = vd_cx.clip(0, 1), vd_cy.clip(0, 1)
ua_cx, ua_cy = ua_cx.clip(0, 1), ua_cy.clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

h1, *_ = axes[0].hist2d(vd_cx, vd_cy, bins=30, cmap="Blues")
plt.colorbar(axes[0].collections[-1], ax=axes[0], label="count")
axes[0].invert_yaxis()
axes[0].set_title("VisDrone — BBox Centre Density", fontweight="bold")
axes[0].set_xlabel("Normalised X centre")
axes[0].set_ylabel("Normalised Y centre (top=0)")

h2, *_ = axes[1].hist2d(ua_cx, ua_cy, bins=30, cmap="Oranges")
plt.colorbar(axes[1].collections[-1], ax=axes[1], label="count")
axes[1].invert_yaxis()
axes[1].set_title("UAVDT — BBox Centre Density", fontweight="bold")
axes[1].set_xlabel("Normalised X centre")
axes[1].set_ylabel("Normalised Y centre (top=0)")

plt.suptitle("Spatial Distribution of BBox Centres (normalised image coordinates)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("spatial_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: spatial_heatmap.png")

**Finding:** VisDrone centres are reasonably spread across the frame with a slight preference for the lower-centre region (roads and footpaths below the horizon). UAVDT shows a more uniform distribution consistent with a bird’s-eye overhead camera. Neither dataset shows a pathological single-point cluster, so we do not need special spatial augmentation to correct for positional bias beyond standard random-flip and mosaic.

---
<a id='section-4'></a>
## Section 4 — Data Quality Issues & Cleaning Decisions

**WHY:** Machine learning models are sensitive to label noise and geometric inconsistencies. We apply five targeted cleaning passes in sequence — each removes a specific class of error, prints before/after counts, and asserts that the fix did what we intended. Cleaning is applied independently to each dataset before merging.

### Issue 1 — Invalid Bounding Boxes

**Problem:** Some annotations have negative coordinates (x < 0 or y < 0) or extend beyond the image boundary (x + w > image_width, y + h > image_height). These cannot be converted to valid normalised YOLO coordinates and would silently corrupt the training targets.

**Standard image sizes assumed:** VisDrone = 1920×1080, UAVDT = 1080×540.

**Decision:** **DROP** any annotation that fails the validity check. Clamping is rejected because it changes the semantic meaning of the box (a partially out-of-frame object becomes artificially truncated).

In [ ]:
# --- Issue 1: Remove bboxes with negative coords or that exceed image boundaries ---
n_vd_0 = len(vd_raw)
n_ua_0 = len(ua_raw)

def invalid_bbox_mask(df, img_w, img_h):
    """Return boolean mask where True = invalid annotation."""
    return (
        (df["bbox_left"]  < 0) |
        (df["bbox_top"]   < 0) |
        (df["bbox_left"] + df["bbox_width"]  > img_w) |
        (df["bbox_top"]  + df["bbox_height"] > img_h)
    )

vd_bad = invalid_bbox_mask(vd_raw, VD_IMG_W, VD_IMG_H)
ua_bad = invalid_bbox_mask(ua_raw, UA_IMG_W, UA_IMG_H)

print("VisDrone invalid breakdown:")
print(f"  Negative x           : {(vd_raw['bbox_left'] < 0).sum()}")
print(f"  Negative y           : {(vd_raw['bbox_top'] < 0).sum()}")
print(f"  Exceeds image width  : {(vd_raw['bbox_left'] + vd_raw['bbox_width'] > VD_IMG_W).sum()}")
print(f"  Exceeds image height : {(vd_raw['bbox_top'] + vd_raw['bbox_height'] > VD_IMG_H).sum()}")
print(f"  Total invalid        : {vd_bad.sum()} / {n_vd_0}")

print("\nUAVDT invalid breakdown:")
print(f"  Negative x           : {(ua_raw['bbox_left'] < 0).sum()}")
print(f"  Negative y           : {(ua_raw['bbox_top'] < 0).sum()}")
print(f"  Exceeds image width  : {(ua_raw['bbox_left'] + ua_raw['bbox_width'] > UA_IMG_W).sum()}")
print(f"  Exceeds image height : {(ua_raw['bbox_top'] + ua_raw['bbox_height'] > UA_IMG_H).sum()}")
print(f"  Total invalid        : {ua_bad.sum()} / {n_ua_0}")

vd_c1 = vd_raw[~vd_bad].copy()
ua_c1 = ua_raw[~ua_bad].copy()

print(f"\nAfter Issue-1 removal:")
print(f"  VisDrone : {n_vd_0:,} → {len(vd_c1):,}  (removed {n_vd_0 - len(vd_c1)})")
print(f"  UAVDT    : {n_ua_0:,} → {len(ua_c1):,}  (removed {n_ua_0 - len(ua_c1)})")

assert (vd_c1["bbox_left"] >= 0).all(),  "Negative x survived cleaning"
assert (vd_c1["bbox_top"]  >= 0).all(),  "Negative y survived cleaning"
assert len(vd_c1) <= n_vd_0,             "Row count increased (bug)"
assert len(ua_c1) <= n_ua_0,             "Row count increased (bug)"
print("\n[OK] All Issue-1 assertions passed.")

### Issue 2 — Degenerate Bounding Boxes

**Problem:** Boxes with width or height below 5 pixels are too small to carry meaningful visual content after resizing to 640×640 (the standard YOLO input). They shrink to sub-pixel noise and act as label noise during training.

**Threshold:** `width < 5 px OR height < 5 px`.

**Decision:** **DROP**. There is no meaningful augmentation that recovers signal from a 1×2-pixel box.

In [ ]:
# --- Issue 2: Remove degenerate bounding boxes (width or height < 5 px) ---
MIN_DIM_PX = 5
n_vd_1 = len(vd_c1)
n_ua_1 = len(ua_c1)

vd_degen = (vd_c1["bbox_width"] < MIN_DIM_PX) | (vd_c1["bbox_height"] < MIN_DIM_PX)
ua_degen = (ua_c1["bbox_width"] < MIN_DIM_PX) | (ua_c1["bbox_height"] < MIN_DIM_PX)

print(f"VisDrone degenerate (w or h < {MIN_DIM_PX} px): {vd_degen.sum()}")
print(f"  Zero-width boxes  : {(vd_c1['bbox_width'] == 0).sum()}")
print(f"  Zero-height boxes : {(vd_c1['bbox_height'] == 0).sum()}")
print(f"\nUAVDT degenerate (w or h < {MIN_DIM_PX} px): {ua_degen.sum()}")
print(f"  Zero-width boxes  : {(ua_c1['bbox_width'] == 0).sum()}")
print(f"  Zero-height boxes : {(ua_c1['bbox_height'] == 0).sum()}")

vd_c2 = vd_c1[~vd_degen].copy()
ua_c2 = ua_c1[~ua_degen].copy()

print(f"\nAfter Issue-2 removal:")
print(f"  VisDrone : {n_vd_1:,} → {len(vd_c2):,}  (removed {n_vd_1 - len(vd_c2)})")
print(f"  UAVDT    : {n_ua_1:,} → {len(ua_c2):,}  (removed {n_ua_1 - len(ua_c2)})")

assert (vd_c2["bbox_width"]  >= MIN_DIM_PX).all(), "Sub-threshold width survived"
assert (vd_c2["bbox_height"] >= MIN_DIM_PX).all(), "Sub-threshold height survived"
assert (ua_c2["bbox_width"]  >= MIN_DIM_PX).all(), "Sub-threshold width survived"
assert (ua_c2["bbox_height"] >= MIN_DIM_PX).all(), "Sub-threshold height survived"
print("\n[OK] All Issue-2 assertions passed.")

### Issue 3 — Outlier Bounding Box Sizes

**Problem:** Even after removing degenerate and out-of-bounds boxes, extremely large bounding boxes remain. These often correspond to annotation errors (e.g., a box drawn around an entire building or the full road surface rather than an individual object).

**Method:** IQR (Interquartile Range) on box **area** (w × h). The upper fence is:
$$\text{upper\_fence} = Q_3 + 1.5 \times IQR$$

We apply only the upper fence (no lower fence) because degenerate small boxes were already removed in Issue 2.

**Decision:** **DROP** any annotation with area above the upper fence. Applied per-dataset to respect the different image scales.

In [ ]:
# --- Issue 3: Remove bbox area outliers using the IQR upper fence ---
def remove_area_outliers(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Drop annotations whose bounding-box area exceeds Q3 + 1.5 * IQR."""
    areas = df["bbox_width"] * df["bbox_height"]
    Q1  = areas.quantile(0.25)
    Q3  = areas.quantile(0.75)
    IQR = Q3 - Q1
    upper_fence = Q3 + 1.5 * IQR
    mask = areas <= upper_fence
    print(f"  {label}:")
    print(f"    Area Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}")
    print(f"    Upper fence = {Q3:.0f} + 1.5 × {IQR:.0f} = {upper_fence:.0f} px²")
    print(f"    Outliers removed: {(~mask).sum()}")
    return df[mask].copy()


n_vd_2 = len(vd_c2)
n_ua_2 = len(ua_c2)

print("IQR upper-fence outlier removal:")
vd_c3 = remove_area_outliers(vd_c2, "VisDrone")
ua_c3 = remove_area_outliers(ua_c2, "UAVDT")

print(f"\nAfter Issue-3 removal:")
print(f"  VisDrone : {n_vd_2:,} → {len(vd_c3):,}  (removed {n_vd_2 - len(vd_c3)})")
print(f"  UAVDT    : {n_ua_2:,} → {len(ua_c3):,}  (removed {n_ua_2 - len(ua_c3)})")

assert len(vd_c3) <= n_vd_2, "Row count increased after IQR filter (bug)"
assert len(ua_c3) <= n_ua_2, "Row count increased after IQR filter (bug)"
print("\n[OK] All Issue-3 assertions passed.")

### Issue 4 — Ignored/Background Class (VisDrone Class 0 and Class 11)

**Problem:**
- **Class 0 (“ignored region”):** VisDrone annotators drew rectangular masks over areas they explicitly excluded from evaluation — construction zones, distant background crowds, parked non-participant vehicles, etc. Keeping these as positive training targets would teach the model to *suppress* detections in those zones rather than detect objects.
- **Class 11 (“others”):** A catch-all for objects that do not fit into the 10 named categories. Without a consistent semantic definition, this class cannot be meaningfully mapped to our 3-class taxonomy.

**Decision:** **DROP** both classes. This is the approach used by the original VisDrone benchmark evaluation toolkit.

In [ ]:
# --- Issue 4: Drop VisDrone ignored-region (class 0) and 'others' (class 11) ---
n_vd_3 = len(vd_c3)

ignored = vd_c3["object_category"] == 0
others  = vd_c3["object_category"] == 11

pct_ignored = ignored.sum() / n_vd_3 * 100
pct_others  = others.sum()  / n_vd_3 * 100

print(f"VisDrone class 0 (ignored region) : {ignored.sum():,} rows  ({pct_ignored:.1f}%)")
print(f"VisDrone class 11 (others)        : {others.sum():,} rows  ({pct_others:.1f}%)")
print(f"Total to drop                     : {(ignored | others).sum():,} rows")

vd_c4 = vd_c3[~(ignored | others)].copy()
ua_c4 = ua_c3.copy()  # UAVDT has no ignored-region class

print(f"\nAfter Issue-4 removal:")
print(f"  VisDrone : {n_vd_3:,} → {len(vd_c4):,}  (removed {n_vd_3 - len(vd_c4)})")
print(f"  UAVDT    : {len(ua_c4):,}  (no change)")
print(f"\nRemaining VisDrone classes: {sorted(vd_c4['object_category'].unique())}")

assert 0  not in vd_c4["object_category"].values, "Class 0 still present"
assert 11 not in vd_c4["object_category"].values, "Class 11 still present"
print("\n[OK] All Issue-4 assertions passed.")

### Issue 5 — Schema Unification

**Problem:** The two datasets use incompatible schemas:
- VisDrone: 12 raw class IDs (0–11), 8 annotation columns, truncation flag
- UAVDT: 3 raw class IDs (1–3), 9 annotation columns (includes MOT tracking fields), no truncation

A single training pipeline cannot consume these directly — we need a shared representation.

**Decision:** Map both datasets to the unified 3-class taxonomy and produce a common DataFrame with columns:
`source, image_id, class_id, class_name, x, y, w, h, truncated, occluded`

UAVDT’s `truncated` field is set to 0 (not annotated in that dataset). MOT tracking columns (`frame_index`, `target_id`, `out_of_view`) are dropped — we treat this as a detection (not tracking) task.

In [ ]:
# --- Issue 5: Map both datasets to the shared 3-class unified schema ---

FINAL_CLASSES = {0: "person", 1: "vehicle", 2: "two_wheeler"}

# VisDrone raw class -> final class (classes 0 and 11 already removed in Issue 4)
VISDRONE_MAP = {
    1: 0,   # pedestrian  -> person
    2: 0,   # people      -> person
    3: 2,   # bicycle     -> two_wheeler
    4: 1,   # car         -> vehicle
    5: 1,   # van         -> vehicle
    6: 1,   # truck       -> vehicle
    7: 2,   # tricycle    -> two_wheeler
    8: 2,   # awning-tri  -> two_wheeler
    9: 1,   # bus         -> vehicle
   10: 2,   # motor       -> two_wheeler
}

# UAVDT raw class -> final class (all vehicle subtypes)
UAVDT_MAP = {1: 1, 2: 1, 3: 1}  # car, truck, bus -> vehicle

UNIFIED_COLS = ["source", "image_id", "class_id", "class_name",
                "x", "y", "w", "h", "truncated", "occluded"]


def unify_visdrone(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["class_id"]   = d["object_category"].map(VISDRONE_MAP)
    d = d.dropna(subset=["class_id"])
    d["class_id"]   = d["class_id"].astype(int)
    d["class_name"] = d["class_id"].map(FINAL_CLASSES)
    return d[["source", "image_id", "class_id", "class_name",
              "bbox_left", "bbox_top", "bbox_width", "bbox_height",
              "truncation", "occlusion"]].rename(columns={
        "bbox_left": "x", "bbox_top": "y",
        "bbox_width": "w", "bbox_height": "h",
        "truncation": "truncated", "occlusion": "occluded",
    })


def unify_uavdt(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["class_id"]   = d["object_category"].map(UAVDT_MAP)
    d = d.dropna(subset=["class_id"])
    d["class_id"]   = d["class_id"].astype(int)
    d["class_name"] = d["class_id"].map(FINAL_CLASSES)
    d["truncated"]  = 0  # not annotated in UAVDT
    return d[["source", "image_id", "class_id", "class_name",
              "bbox_left", "bbox_top", "bbox_width", "bbox_height",
              "truncated", "occlusion"]].rename(columns={
        "bbox_left": "x", "bbox_top": "y",
        "bbox_width": "w", "bbox_height": "h",
        "occlusion": "occluded",
    })


vd_unified = unify_visdrone(vd_c4)
ua_unified = unify_uavdt(ua_c4)

print("Unified VisDrone schema:")
display(vd_unified.head(4))
print(f"  Shape: {vd_unified.shape}")
print(f"  Classes: {vd_unified['class_name'].value_counts().to_dict()}")

print("\nUnified UAVDT schema:")
display(ua_unified.head(4))
print(f"  Shape: {ua_unified.shape}")
print(f"  Classes: {ua_unified['class_name'].value_counts().to_dict()}")

expected = set(UNIFIED_COLS)
assert set(vd_unified.columns) == expected, f"VD schema mismatch: {set(vd_unified.columns)}"
assert set(ua_unified.columns) == expected, f"UA schema mismatch: {set(ua_unified.columns)}"
assert vd_unified["class_id"].between(0, 2).all(), "Out-of-range class_id in VD"
assert ua_unified["class_id"].between(0, 2).all(), "Out-of-range class_id in UA"
print("\n[OK] All Issue-5 assertions passed. Schemas are aligned.")

---
<a id='section-5'></a>
## Section 5 — Merged Dataset Summary

**WHY:** With both datasets cleaned and schema-unified, we merge them into a single DataFrame representing the full training corpus. This section verifies the final counts, visualises the class balance across sources, and prints a structured quality report.

In [ ]:
# --- Combine both cleaned, schema-unified datasets into one DataFrame ---
merged = pd.concat([vd_unified, ua_unified], ignore_index=True)
merged = merged.sort_values(["source", "image_id"]).reset_index(drop=True)

print(f"Merged dataset shape  : {merged.shape}")
print(f"Unique images         : {merged['image_id'].nunique():,}")
print(f"  VisDrone images     : {merged[merged.source == 'VisDrone']['image_id'].nunique():,}")
print(f"  UAVDT images        : {merged[merged.source == 'UAVDT']['image_id'].nunique():,}")
print()
display(merged.head(10))

assert merged["source"].isin(["VisDrone", "UAVDT"]).all(), "Unknown source in merged df"
assert merged["class_id"].between(0, 2).all(),             "Out-of-range class_id in merged df"
print("[OK] Merge assertions passed.")

In [ ]:
# --- Final class distribution across the merged dataset (per source + combined) ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sources = [("VisDrone", axes[0]), ("UAVDT", axes[1])]
for src, ax in sources:
    sub = merged[merged["source"] == src]
    counts = sub["class_name"].value_counts()
    colors = [CLASS_PALETTE.get(c, "#888") for c in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v,
                f"{v:,}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"{src} — Final Distribution", fontweight="bold")
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")

# Combined
all_counts = merged["class_name"].value_counts()
colors = [CLASS_PALETTE.get(c, "#888") for c in all_counts.index]
bars = axes[2].bar(all_counts.index, all_counts.values, color=colors)
for bar, v in zip(bars, all_counts.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, v,
                 f"{v:,}\n({v/len(merged)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=8)
axes[2].set_title("Combined — Final Distribution", fontweight="bold")
axes[2].set_xlabel("Class")
axes[2].set_ylabel("Count")

# Legend
patches = [mpatches.Patch(color=v, label=k) for k, v in CLASS_PALETTE.items()]
fig.legend(handles=patches, loc="upper right", fontsize=9, title="Class")

plt.suptitle("Final Class Distribution After Cleaning and Schema Unification",
             fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("final_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: final_class_distribution.png")

In [ ]:
# --- Final data quality report: comprehensive before/after summary ---
n_raw_vd    = len(vd_raw)
n_raw_ua    = len(ua_raw)
n_final_vd  = len(vd_unified)
n_final_ua  = len(ua_unified)
n_raw_total = n_raw_vd + n_raw_ua
n_final     = len(merged)
n_removed   = n_raw_total - n_final
pct_removed = n_removed / n_raw_total * 100

# Per-cleaning-step counts
steps_vd = [
    ("Raw input",                 n_raw_vd),
    ("After invalid bbox removal", len(vd_c1)),
    ("After degenerate removal",   len(vd_c2)),
    ("After IQR outlier removal",  len(vd_c3)),
    ("After ignored-class drop",   len(vd_c4)),
    ("After schema unification",   n_final_vd),
]
steps_ua = [
    ("Raw input",                 n_raw_ua),
    ("After invalid bbox removal", len(ua_c1)),
    ("After degenerate removal",   len(ua_c2)),
    ("After IQR outlier removal",  len(ua_c3)),
    ("After ignored-class drop",   len(ua_c4)),
    ("After schema unification",   n_final_ua),
]

print("=" * 62)
print("  FINAL DATA QUALITY REPORT")
print("=" * 62)

print("\nVisDrone cleaning pipeline:")
for step, n in steps_vd:
    print(f"  {step:<35} {n:>8,}")

print("\nUAVDT cleaning pipeline:")
for step, n in steps_ua:
    print(f"  {step:<35} {n:>8,}")

print("\nMerged dataset:")
print(f"  {'Total raw annotations':<35} {n_raw_total:>8,}")
print(f"  {'Total final annotations':<35} {n_final:>8,}")
print(f"  {'Annotations removed':<35} {n_removed:>8,}  ({pct_removed:.1f}%)")
print(f"  {'Unique images':<35} {merged['image_id'].nunique():>8,}")

print("\nFinal class breakdown:")
for cls_name, cnt in merged["class_name"].value_counts().items():
    pct = cnt / n_final * 100
    print(f"  {cls_name:<35} {cnt:>8,}  ({pct:.1f}%)")

print("=" * 62)

---
<a id='section-6'></a>
## Section 6 — Summary / Data Card

| Attribute | VisDrone | UAVDT | Combined |
|-----------|----------|-------|----------|
| **Source** | Google Drive (academic release) | Zenodo record 14575517 | — |
| **Acquisition** | `scripts/download_visdrone.py` | `scripts/download_uavdt.py` | — |
| **Raw annotation format** | 8-field CSV per line (x,y,w,h,score,class,trunc,occ) | 9-field MOT CSV (frame,tid,x,y,w,h,oov,occ,class) | unified schema |
| **Image resolution** | 1920×1080 | 1080×540 | mixed |
| **Raw annotations** | ~540 000 (full) | ~840 000 (full) | ~1.38 M |
| **Final classes** | person, vehicle, two_wheeler | vehicle only | person, vehicle, two_wheeler |
| **Class IDs** | 0, 1, 2 | 0, 1, 2 | 0=person, 1=vehicle, 2=two_wheeler |

### Cleaning decisions

| Step | Rule | Reason |
|------|------|--------|
| 1. Invalid bboxes | Drop if x<0, y<0, x+w>W, y+h>H | Cannot produce valid YOLO coordinates |
| 2. Degenerate boxes | Drop if w<5 px or h<5 px | Sub-pixel after resize; act as noise |
| 3. Area outliers | Drop if area > Q3+1.5×IQR | Likely annotation errors (whole-road boxes) |
| 4. Ignored class | Drop VisDrone class 0 and class 11 | No learnable signal; inconsistent definition |
| 5. Schema unification | Remap raw class IDs; rename columns | Enable single-pipeline training |

### Design decisions

- **Retained all occlusion levels.** Excluding occluded objects would make the training distribution unrealistically easy for drone scenes where occlusion is the norm.
- **UAVDT `truncated` set to 0.** The field is not annotated in that dataset. This slight schema asymmetry is documented here so downstream models can weight the field appropriately.
- **No clamping.** Out-of-bounds coordinates are dropped rather than clamped. Clamping changes the semantic meaning of the box and introduces implicit truncation annotations.
- **IQR applied per-dataset.** UAVDT and VisDrone have different image resolutions, so applying a single global IQR across both would be biased toward the larger image scale.
- **Synthetic fallback.** When raw data is not present, the notebook generates statistically plausible synthetic data so that all code cells run end-to-end. Synthetic mode is clearly flagged in every relevant output.

### Next steps (Step 6 — Model Baseline)
- Convert the unified DataFrame to YOLO-format `.txt` label files using `src/data_processing/build_all.py`
- Train a YOLOv8-nano baseline on each of the three dataset variants (VisDrone-only, UAVDT-only, combined)
- Compare mAP@50 across variants to quantify the multi-source benefit